# ArgumentProfile : la fiche d'identité multidimensionnelle d'un argument

Ce carnet démontre `ArgumentProfile`, la **vue agrégée** de toutes les analyses portant sur un même
argument. Il complète [Argumentation-02-Fallacies-Detection-Python](Argumentation-02-Fallacies-Detection-Python.ipynb)
(détection de sophismes par taxonomie, qui couvre la *dimension sophismes*) en
réunissant les **cinq dimensions** de l'analyse sur un même objet : sophismes,
qualité, contre-arguments, croyances JTMS et résultats formels.

Le carnet a deux parties. La **baseline** remplit l'état **à la main** : un comparateur
déterministe, sans service, qui fixe ce que chaque dimension contient et ce que le profil en
restitue. La **seconde partie** refait le même travail avec les agents de `argumentation_lib`,
puis compare les deux — dimension par dimension, en nommant ce que les agents produisent
aujourd'hui et ce qu'ils ne produisent pas encore.

**Durée estimée** : 50 minutes
**Prérequis** : [Argumentation-02-Fallacies-Detection-Python](Argumentation-02-Fallacies-Detection-Python.ipynb) (taxonomie des sophismes) et, pour la seconde partie, un service LLM OpenAI-compatible (voir section 6).

## Objectifs

- Comprendre pourquoi l'analyse argumentative est **multidimensionnelle** : un même argument peut être fallacieux (logique informelle), mal fondé (qualité), attaqué (contre-argumentation), non croyable (JTMS) et formellement invalide (logique formelle)
- Construire un `ArgumentProfile` : la fiche agrégée qui réunit, pour **un** argument, toutes les analyses produites par les dimensions
- Utiliser les méthodes cross-argument (`get_weak_arguments`, `get_fallacious_arguments`) pour trier un débat entier
- **Comparer** une table injectée et un remplissage par les agents, et distinguer une dimension *produite* d'une dimension seulement *représentée*
- Mesurer ce qu'un verdict multidimensionnel doit au nombre de dimensions qu'il lit réellement

**Série Argumentation** | | Précédent : [08d-Restitution-3-Actes-Python](./Argumentation-08d-Restitution-3-Actes-Python.ipynb) | Suivant : [08f-Synthese-Multidimensionnelle-Python](./Argumentation-08f-Synthese-Multidimensionnelle-Python.ipynb) | [↑ Série Argumentation](README.md)


## 1. Pourquoi un profil par argument ?

Une analyse rhétorique complète produit plusieurs types de résultats — sophismes détectés, scores de qualité, contre-arguments générés, croyances d'un réseau JTMS, verdicts formels. Ces résultats vivent naturellement dans des **structures séparées** de l'état partagé (un dict de sophismes, un dict de scores, une liste de contre-arguments...).

Le problème : pour **raisonner sur un argument précis**, il faut réunir tout ce qui le concerne. *« Cet argument est-il fallacieux ET mal fondé ET attaqué ? »* C'est exactement le rôle de `ArgumentProfile` : une **vue projetée** qui agrège, pour un `arg_id` donné, toutes les analyses qui le ciblent.

| Dimension | Source dans l'état | Question répondue |
|-----------|--------------------|--------------------|
| **Sophismes** | `identified_fallacies` (target) | L'argument commet-il une erreur rhétorique ? |
| **Qualité** | `argument_quality_scores` | Est-il clair, bien fondé, pertinent ? |
| **Contre-arguments** | `counter_arguments` (target) | Comment peut-on l'attaquer ? |
| **Croyances JTMS** | `jtms_beliefs` | Est-il jugé valide dans le réseau de croyances ? |
| **Résultats formels** | `propositional_analysis_results` / `fol_analysis_results` (par `arg_id`) | Est-il logiquement valide ? |

Ces cinq verdicts sont **indépendants par conception** : un argument peut être formellement valide (pas d'erreur logique) et rhétoriquement fallacieux (appel à l'autorité). Le profil est la seule vue qui les réunit.

Le carnet sépare deux choses que cette formule confond souvent. **Ce qui est écrit dans l'état** se mesure : la section 3 le restitue argument par argument, en imprimant pour chacun combien de dimensions ont réellement été lues. **L'indépendance elle-même** ne se déduit pas de trois arguments : la section 5 la traite comme une hypothèse et mesure ce qu'un échantillon aussi petit peut en dire. La section 6 change de producteur — les agents de la lib au lieu d'appels écrits à la main — et situe la frontière de ce qu'ils remplissent aujourd'hui.


In [1]:
# --- Imports + texte du debat (deterministe, auto-contenu) ---
import sys, logging
from pathlib import Path

# Imports de la lib portee (argumentation_lib, cote a cote des notebooks).
sys.path.insert(0, str(Path(".").resolve()))
logging.disable(logging.INFO)  # taire les logs de mutation pour lisibilite

from argumentation_lib import UnifiedAnalysisState, ArgumentProfile

# Un debat court, construit pour exercer les 5 dimensions (FR, deterministe).
DEBAT = (
    "Voici mon raisonnement. D'abord, tous les experts s'accordent a dire que le projet "
    "est utile, donc il faut le financer. Ensuite, soit on le finance, soit on abandonne "
    "toute innovation. Enfin, les couts sont eleves, mais le benefice attendu justifie "
    "l'investissement."
)
print("Texte du debat (auto-contenu, deterministe) :")
print(DEBAT)
print(f"\nargumentation_lib importe. ArgumentProfile = {ArgumentProfile.__name__} (dataclass agreggee).")


Texte du debat (auto-contenu, deterministe) :
Voici mon raisonnement. D'abord, tous les experts s'accordent a dire que le projet est utile, donc il faut le financer. Ensuite, soit on le finance, soit on abandonne toute innovation. Enfin, les couts sont eleves, mais le benefice attendu justifie l'investissement.

argumentation_lib importe. ArgumentProfile = ArgumentProfile (dataclass agreggee).


## 2. L'état partagé : registre central des analyses

On instancie `UnifiedAnalysisState` avec le texte brut. C'est le **registre central** où chaque dimension viendra écrire ses résultats. Les arguments identifiés reçoivent un `arg_id` (`arg_1`, `arg_2`...) — la clé qui reliera tout.


In [2]:
# --- Etat partage + identification des arguments : LA BASELINE ---
# Jusqu'a la section 5 incluse, tout ce qui entre dans `etat` est ecrit a la main : c'est la
# baseline, deterministe et sans service. Elle fixe le contrat du profil -- ce que chaque
# dimension contient, ce que le profil en restitue. La section 6 refait le meme travail avec
# les agents de la lib, sur un etat SEPARE (`etat_agent`), puis compare les deux.
etat = UnifiedAnalysisState(DEBAT)

# Trois arguments extraits du texte (en pratique par un extracteur, ici deterministe).
a_expert = etat.add_argument("Tous les experts s'accordent => le projet est utile (appeal to authority + hasty generalization)")
a_fauxdilemme = etat.add_argument("Soit on finance, soit on abandonne toute innovation (faux dilemme)")
a_cout = etat.add_argument("Les couts sont eleves mais le benefice justifie l'investissement (balance cout-benefice)")

print("Arguments identifies :")
for aid, desc in etat.identified_arguments.items():
    print(f"  {aid}: {desc[:70]}...")


Arguments identifies :
  arg_1: Tous les experts s'accordent => le projet est utile (appeal to authori...
  arg_2: Soit on finance, soit on abandonne toute innovation (faux dilemme)...
  arg_3: Les couts sont eleves mais le benefice justifie l'investissement (bala...


### Dimension 1 — Sophismes (lien avec [Argumentation-02-Fallacies-Detection-Python](Argumentation-02-Fallacies-Detection-Python.ipynb))

On attache des sophismes aux arguments, en suivant la **taxonomie en familles** (cf. Agentic-1-informal). Chaque sophisme cible un `arg_id` précis.


In [3]:
# --- Dimension 1 : sophismes (taxonomie familles) ---
etat.add_fallacy("appel_a_l_autorite", "aucun expert n'est nomme ni cite",
                 target_arg_id=a_expert, family="Obstruction", taxonomy_path="fallacies/obstruction/appeal_authority")
etat.add_fallacy("generalisation_hative", "tous les experts' sans quantifier ni echantillon",
                 target_arg_id=a_expert, family="Obstruction")
etat.add_fallacy("faux_dilemme", "tiers exclu (co-partage, re-echelonnage, prototypage...)",
                 target_arg_id=a_fauxdilemme, family="Obstruction")

print(f"Sophismes identifies : {len(etat.identified_fallacies)}")
for fid, info in etat.identified_fallacies.items():
    print(f"  {fid} [{info.get('family','?')}] {info['type']} -> cible {info.get('target_argument_id','?')}")


Sophismes identifies : 3
  fallacy_1 [Obstruction] appel_a_l_autorite -> cible arg_1
  fallacy_2 [Obstruction] generalisation_hative -> cible arg_1
  fallacy_3 [Obstruction] faux_dilemme -> cible arg_2


### Dimension 2 — Qualité (9 vertus, agrégées en note globale)

Chaque argument est noté sur un sous-ensemble des **vertus** (clarté, fondement, pertinence...), chaque vertu dans **[0, 1]**. Le score `overall` est la **somme** des vertus évaluées (contrat d'unité #1942 — pas une note sur 10) ; la fraction lisible vaut `overall / len(scores)`, et le seuil faible de référence est **0.5**.


In [4]:
# --- Dimension 2 : scores de qualite (vertus) ---
# Contrat d'unite (#1942) : vertus dans [0, 1], overall = SOMME des vertus evaluees.
etat.add_quality_score(a_expert, scores={"clarte": 0.6, "fondement": 0.2, "pertinence": 0.5}, overall=1.3)
etat.add_quality_score(a_fauxdilemme, scores={"clarte": 0.7, "fondement": 0.25, "pertinence": 0.6}, overall=1.55)
etat.add_quality_score(a_cout, scores={"clarte": 0.8, "fondement": 0.7, "pertinence": 0.8}, overall=2.3)

print("Scores de qualite :")
for aid, sc in etat.argument_quality_scores.items():
    fraction = sc['overall'] / len(sc['scores'])
    print(f"  {aid}: overall={sc['overall']:.2f} (somme)  fraction={fraction:.3f}  vertus={sc['scores']}")


Scores de qualite :
  arg_1: overall=1.30 (somme)  fraction=0.433  vertus={'clarte': 0.6, 'fondement': 0.2, 'pertinence': 0.5}
  arg_2: overall=1.55 (somme)  fraction=0.517  vertus={'clarte': 0.7, 'fondement': 0.25, 'pertinence': 0.6}
  arg_3: overall=2.30 (somme)  fraction=0.767  vertus={'clarte': 0.8, 'fondement': 0.7, 'pertinence': 0.8}


### Dimension 3 — Contre-arguments (5 stratégies)

Pour chaque argument attaqué, on génère des contre-arguments selon des stratégies canoniques (contester la source, l'inférence, la portée...). Chaque contre-argument a un `score` de force.


In [5]:
# --- Dimension 3 : contre-arguments (strategies) ---
etat.add_counter_argument(original_arg="experts s'accordent",
                          counter_content="aucun expert nomme => source inexistante",
                          strategy="contester_source", score=0.85, target_arg_id=a_expert)
etat.add_counter_argument(original_arg="tous les experts",
                          counter_content="aucun sondage => generalisation non etayee",
                          strategy="contester_inference", score=0.75, target_arg_id=a_expert)
etat.add_counter_argument(original_arg="soit... soit...",
                          counter_content="tiers exclu : co-partage, re-echelonnage, prototypage",
                          strategy="denoncer_faux_dilemme", score=0.90, target_arg_id=a_fauxdilemme)

print(f"Contre-arguments generes : {len(etat.counter_arguments)}")
for ca in etat.counter_arguments:
    print(f"  {ca['id']} [{ca['strategy']}] force={ca['score']:.2f} -> cible {ca.get('target_arg_id','?')}")


Contre-arguments generes : 3
  ca_1 [contester_source] force=0.85 -> cible arg_1
  ca_2 [contester_inference] force=0.75 -> cible arg_1
  ca_3 [denoncer_faux_dilemme] force=0.90 -> cible arg_2


### Dimension 4 — Croyances JTMS (réseau de justification)

Le **JTMS** (Justification-Based Truth Maintenance System) trace quelles croyances sont valides/invalides et pourquoi. Une croyance se **rattache à un argument** quand son nom reprend le texte de l'argument (le profil apparie sur le préfixe de la description) — c'est la convention de projection qui lie une croyance JTMS débat-large à un argument précis.


In [6]:
# --- Dimension 4 : croyances JTMS ---
# Le nom reprend le debut de l'argument -> la croyance se rattache au profil de cet argument.
etat.add_jtms_belief(name="Tous les experts s'accordent => le projet est utile", valid=False,
                     justifications=["autorite_non_citee", "generalisation_non_quantifiee"])
etat.add_jtms_belief(name="Les couts sont eleves mais le benefice justifie l'investissement", valid=True,
                     justifications=["benefice > cout (balance)"])

print(f"Croyances JTMS : {len(etat.jtms_beliefs)}")
for bid, b in etat.jtms_beliefs.items():
    print(f"  {bid}: {b['name'][:55]}... valid={b['valid']}")
print("\nConvention : le nom reprenant le texte de l'argument, la croyance se rattache")
print("au profil de cet argument (match par prefixe de description).")


Croyances JTMS : 2
  jtms_1: Tous les experts s'accordent => le projet est utile... valid=False
  jtms_2: Les couts sont eleves mais le benefice justifie l'inves... valid=True

Convention : le nom reprenant le texte de l'argument, la croyance se rattache
au profil de cet argument (match par prefixe de description).


### Dimension 5 — Résultats formels (logique PL/FOL)

La dimension formelle agrège les verdicts des solveurs logiques. `get_argument_profile` lit les résultats **liés à un argument** (`arg_id=`) : `add_propositional_analysis_result` (PL), `add_fol_analysis_result` (FOL) et `add_nl_to_logic_translation`. Un argument peut être formellement satisfiable (pas de contradiction interne) tout en étant fallacieux : les deux dimensions disent deux choses différentes d'un même argument, et le profil les tient l'une à côté de l'autre.

> **Note** : `add_formal_synthesis_report` existe aussi, mais produit une **synthèse débat-large** (non liée à un `arg_id`) que le profil, par conception, ne projette pas. La synthèse globale et l'analyse par-argument sont deux objets distincts.


In [7]:
# --- Dimension 5 : resultats formels par argument (PL + FOL) ---
etat.add_propositional_analysis_result(
    formulas=["Expert => Utile", "Expert", "Utile"], satisfiable=True,
    model={"Expert": True, "Utile": True}, arg_id=a_expert)
etat.add_fol_analysis_result(
    formulas=["forall x. Expert(x) => Utile(x)"], consistent=True,
    inferences=["Utile(projet)"], confidence=0.4, arg_id=a_expert)
etat.add_propositional_analysis_result(
    formulas=["Finance OU Abandon", "non Abandon"], satisfiable=True,
    model={"Finance": True, "Abandon": False}, arg_id=a_fauxdilemme)

print("Resultats formels par argument :")
for aid in [a_expert, a_fauxdilemme, a_cout]:
    pl = [r for r in etat.propositional_analysis_results if r.get('arg_id') == aid]
    fol = [r for r in etat.fol_analysis_results if r.get('arg_id') == aid]
    print(f"  {aid}: PL={len(pl)} FOL={len(fol)}")


Resultats formels par argument :
  arg_1: PL=1 FOL=1
  arg_2: PL=1 FOL=0
  arg_3: PL=0 FOL=0


## 3. Le profil : réunir les cinq dimensions pour UN argument

C'est le cœur du notebook. `get_argument_profile(arg_id)` **projette** l'état partagé sur un seul argument : il récupère les sophismes qui le ciblent, son score de qualité, ses contre-arguments, etc. On obtient une **fiche d'identité** exploitable pour le raisonnement.

Regardons le profil de `a_expert` — l'argument le plus attaqué du débat.


In [8]:
# --- Le profil agregge d'un argument (la valeur centrale) ---
profil_expert = etat.get_argument_profile(a_expert)

print("=" * 64)
print(f"PROFIL DE {profil_expert.arg_id}")
print("=" * 64)
print(f"Description       : {profil_expert.description[:70]}...")
print(f"Sophismes (dim 1) : {len(profil_expert.fallacies)}  ->  {[f.get('type') for f in profil_expert.fallacies]}")
if profil_expert.quality_score:
    _q = profil_expert.quality_score
    print(f"Qualite   (dim 2) : overall={_q['overall']:.2f} (somme)  fraction={_q['overall'] / len(_q['scores']):.3f}  vertus={_q['scores']}")
print(f"Contre-arg(dim 3) : {len(profil_expert.counter_arguments)}  strategies={[c.get('strategy') for c in profil_expert.counter_arguments]}")
print(f"JTMS      (dim 4) : {len(profil_expert.jtms_beliefs)} croyances attachees")
print(f"Formel    (dim 5) : {len(profil_expert.formal_results)} synthese(s)")
print()
print("LECTURE : cet argument est fallacieux (2 sophismes), mal fonde (fraction 1.3/3 ~ 0.43 < 0.5),")
print("fortement attaque (2 contre-arguments, force moy ~0.80) -> profil COHERENT")
print("de faiblesse multidimensionnelle. Aucune dimension ne le defend.")


PROFIL DE arg_1
Description       : Tous les experts s'accordent => le projet est utile (appeal to authori...
Sophismes (dim 1) : 2  ->  ['appel_a_l_autorite', 'generalisation_hative']
Qualite   (dim 2) : overall=1.30 (somme)  fraction=0.433  vertus={'clarte': 0.6, 'fondement': 0.2, 'pertinence': 0.5}
Contre-arg(dim 3) : 2  strategies=['contester_source', 'contester_inference']
JTMS      (dim 4) : 1 croyances attachees
Formel    (dim 5) : 2 synthese(s)

LECTURE : cet argument est fallacieux (2 sophismes), mal fonde (fraction 1.3/3 ~ 0.43 < 0.5),
fortement attaque (2 contre-arguments, force moy ~0.80) -> profil COHERENT
de faiblesse multidimensionnelle. Aucune dimension ne le defend.


### Verdict multidimensionnel : lire ce que les dimensions disent

Comparons les trois profils, cette fois sur les **cinq dimensions** et non sur deux. Un verdict « FAIBLE » signifie que **deux dimensions au moins** sont en défaut ; « SOLIDE », qu'aucune ne l'est **et** qu'au moins trois dimensions ont été lues ; « MIXTE », tout le reste.

Deux seuils entrent dans ces défauts, écrits en tête de la cellule : une qualité est en défaut sous la fraction 0,5 — le seuil de référence de la lib — et une attaque l'est quand un contre-argument atteint 0,7. Le second est un choix de ce carnet, que la lib n'impose pas : un argument dont le contre-argument le plus fort approche cette valeur voit son verdict en dépendre. La colonne `attq` donne cette force maximale.

La colonne `couv` compte ces dimensions lues. Elle n'est pas décorative : un argument jugé sur deux axes et un argument jugé sur cinq ne devraient pas porter le même mot sans qu'on puisse le voir.

Le profil ne fabrique donc pas un nombre unique à partir de cinq signaux hétérogènes. Il rend un verdict **avec ses appuis** — la liste des dimensions en défaut, et le nombre de dimensions réellement consultées.


In [9]:
# --- Verdict multidimensionnel : cinq signaux, et la couverture dit ce qu'il lit ---
SEUIL_QUALITE = 0.5        # fraction overall / len(scores) : en dessous, qualité faible
SEUIL_ATTAQUE_FORTE = 0.7  # score d'un contre-argument jugé fort


def signaux(profil):
    """Extrait les cinq signaux d'un profil. None = dimension non produite pour cet argument."""
    sig = {"sophismes": len(profil.fallacies)}

    q = profil.quality_score
    # `float` : l'ecrivain d'etat accepte une chaine et la stocke telle quelle ; le lecteur, lui, calcule.
    sig["qualite"] = (float(q["overall"]) / len(q["scores"])) if q and q.get("scores") else None

    sig["attaque"] = max((float(c.get("score", 0.0)) for c in profil.counter_arguments), default=None)

    jtms = [b.get("valid") for b in profil.jtms_beliefs]
    sig["jtms"] = (False in jtms) if jtms else None

    verdicts = []
    for r in profil.formal_results:
        if "satisfiable" in r:
            verdicts.append(bool(r["satisfiable"]))
        elif "consistent" in r:
            verdicts.append(bool(r["consistent"]))
    sig["formel"] = (False in verdicts) if verdicts else None
    return sig


def defauts(sig):
    """Dimensions en défaut, parmi celles qui sont réellement produites pour cet argument."""
    d = []
    if sig["sophismes"] >= 1:
        d.append("sophismes")
    if sig["qualite"] is not None and sig["qualite"] < SEUIL_QUALITE:
        d.append("qualite")
    if sig["attaque"] is not None and sig["attaque"] >= SEUIL_ATTAQUE_FORTE:
        d.append("attaque")
    if sig["jtms"] is True:
        d.append("jtms")
    if sig["formel"] is True:
        d.append("formel")
    return d


def verdict_multidim(profil):
    """Verdict qui ne compte que les dimensions produites.

    FAIBLE : deux défauts ou plus · SOLIDE : aucun défaut et au moins trois dimensions lues ·
    MIXTE : tout le reste. La couverture est rendue avec le verdict : un verdict sur deux
    dimensions n'a pas la même portée qu'un verdict sur cinq.
    """
    sig = signaux(profil)
    d = defauts(sig)
    couverture = sum(1 for v in sig.values() if v is not None)
    if len(d) >= 2:
        v = "FAIBLE"
    elif not d and couverture >= 3:
        v = "SOLIDE"
    else:
        v = "MIXTE"
    return v, d, couverture, sig


def _txt(sig, cle):
    v = sig[cle]
    if v is None:
        return "-"
    if cle in ("jtms", "formel"):
        return "défaut" if v is True else "ok"
    if cle in ("qualite", "attaque"):
        return f"{v:.2f}"
    return str(v)


print(f"{'arg':>8} {'soph':>5} {'qual':>5} {'attq':>6} {'jtms':>7} {'form':>7} {'couv':>5}  verdict (dimensions en défaut)")
print("-" * 108)
bilan_baseline = {}
for aid in [a_expert, a_fauxdilemme, a_cout]:
    v, d, couv, sig = verdict_multidim(etat.get_argument_profile(aid))
    bilan_baseline[aid] = (v, d, couv, sig)
    print(
        f"{aid:>8} {sig['sophismes']:>5} {_txt(sig, 'qualite'):>5} {_txt(sig, 'attaque'):>6} "
        f"{_txt(sig, 'jtms'):>7} {_txt(sig, 'formel'):>7} {couv:>5}  {v} ({', '.join(d) or 'aucun défaut'})"
    )
print()
print("`couv` = nombre de dimensions réellement lues pour cet argument. Un verdict rendu sur trois")
print("dimensions ne dit rien des deux autres : il se lit avec sa couverture, jamais seul.")


     arg  soph  qual   attq    jtms    form  couv  verdict (dimensions en défaut)
------------------------------------------------------------------------------------------------------------
   arg_1     2  0.43   0.85  défaut      ok     5  FAIBLE (sophismes, qualite, attaque, jtms)
   arg_2     1  0.52   0.90       -      ok     4  FAIBLE (sophismes, attaque)
   arg_3     0  0.77      -      ok       -     3  SOLIDE (aucun défaut)

`couv` = nombre de dimensions réellement lues pour cet argument. Un verdict rendu sur trois
dimensions ne dit rien des deux autres : il se lit avec sa couverture, jamais seul.


## 4. Trier le débat entier : analyses cross-argument

Le profil n'est pas qu'une vue passive — il alimente des **requêtes cross-argument** : quels sont les arguments faibles du débat ? Lesquels sont fallacieux ? Ces méthodes retournent des listes de profils, prêtes pour la priorisation (faut-il réviser les arguments faibles d'abord ? Les écarter ?).


In [10]:
# --- Analyses cross-argument (trier le debat) ---
faibles = etat.get_weak_arguments(threshold=0.5)
fallacieux = etat.get_fallacious_arguments()

print(f"Arguments FAIBLES (fraction < 0.5) : {len(faibles)}")
for p in faibles:
    _q = p.quality_score
    print(f"  - {p.arg_id}: fraction={_q['overall'] / len(_q['scores']):.3f} (overall={_q['overall']:.2f})")

print(f"\nArguments FALLACIEUX (>= 1 sophisme) : {len(fallacieux)}")
for p in fallacieux:
    types = [f.get('type') for f in p.fallacies]
    print(f"  - {p.arg_id}: {types}")

print("\nUtilisation : un agent peut prioriser la revision des arguments faibles,")
print("ecarter les fallacieux de la conclusion, ou cibler les contre-arguments les plus forts.")
print("\nResume agregge (get_enrichment_summary) :")
import json as _json
print(_json.dumps(etat.get_enrichment_summary(), indent=2, ensure_ascii=False)[:600])


Arguments FAIBLES (fraction < 0.5) : 1
  - arg_1: fraction=0.433 (overall=1.30)

Arguments FALLACIEUX (>= 1 sophisme) : 2
  - arg_1: ['appel_a_l_autorite', 'generalisation_hative']
  - arg_2: ['faux_dilemme']

Utilisation : un agent peut prioriser la revision des arguments faibles,
ecarter les fallacieux de la conclusion, ou cibler les contre-arguments les plus forts.

Resume agregge (get_enrichment_summary) :
{
  "total_arguments": 3,
  "with_fallacy_analysis": 2,
  "with_quality_score": 3,
  "with_counter_argument": 2,
  "with_formal_verification": 2,
  "with_jtms_belief": 2,
  "gaps": [
    "arg_3 has no formal verification"
  ]
}


## 5. L'indépendance : hypothèse de conception ou résultat ?

Les sections précédentes ont rempli l'état **à la main**. C'était nécessaire pour fixer le contrat du profil, mais une affirmation a accompagné tout ce travail sans être testée : *les dimensions sont indépendantes*.

Une indépendance se mesure. Si deux dimensions sont indépendantes, connaître l'une ne dit rien de l'autre : les arguments où l'une est en défaut et l'autre non doivent donc être fréquents. Compter ces co-occurrences, sur tous les arguments dont on dispose, est la seule façon de confronter la formule aux données — et le **nombre** de ces arguments est le vrai juge du résultat.

**Comment lire le tableau.** Chaque ligne porte une paire de dimensions. Les colonnes `deux`, `A seul`, `B seul` et `aucun` partitionnent les arguments pour lesquels les deux dimensions de la paire sont renseignées : ceux où les deux sont en défaut, ceux où une seule l'est, et ceux où aucune ne l'est. La colonne `lus` est la **somme de ces quatre cases** — le nombre d'arguments réellement lisibles sur cette paire, `aucun` compris. C'est donc un compte de lisibilité, pas un compte de défauts : `lus` vaut `deux + A seul + B seul + aucun`, et dépasse souvent `deux`. Un argument dont une des deux dimensions n'a pas été produite est écarté de la paire et n'entre dans aucune de ces cases ; c'est pourquoi `lus` reste inférieur ou égal au nombre d'arguments de l'échantillon, sans lui être toujours égal.


In [11]:
# --- L'indépendance des dimensions : ce qu'on peut mesurer avec n arguments ---
DIMENSIONS = ["sophismes", "qualite", "attaque", "jtms", "formel"]


def en_defaut(sig, cle):
    """Le signal est-il en défaut sur cette dimension ? None (non produite) n'est ni vrai ni faux."""
    v = sig[cle]
    if v is None:
        return None
    if cle in ("jtms", "formel"):
        return v is True
    if cle == "sophismes":
        return v >= 1
    if cle == "qualite":
        return v < SEUIL_QUALITE
    return v >= SEUIL_ATTAQUE_FORTE


def matrice_cooccurrence(profils, etiquette):
    """Pour chaque paire de dimensions : combien d'arguments portent les deux défauts, un seul,
    ou aucun des deux. Les arguments où une des deux dimensions manque sont écartés de la paire."""
    lignes = [signaux(p) for p in profils]
    print(f"Co-occurrence des défauts — {etiquette} (n = {len(lignes)} arguments)")
    print(f"{'paire':>24} {'lus':>5} {'deux':>5} {'A seul':>7} {'B seul':>7} {'aucun':>6}")
    print("-" * 62)
    for i, a in enumerate(DIMENSIONS):
        for b in DIMENSIONS[i + 1:]:
            deux = seul_a = seul_b = aucun = 0
            for sig in lignes:
                da, db = en_defaut(sig, a), en_defaut(sig, b)
                if da is None or db is None:
                    continue
                if da and db:
                    deux += 1
                elif da:
                    seul_a += 1
                elif db:
                    seul_b += 1
                else:
                    aucun += 1
            n_lus = deux + seul_a + seul_b + aucun
            if n_lus:
                print(f"{a + ' / ' + b:>24} {n_lus:>5} {deux:>5} {seul_a:>7} {seul_b:>7} {aucun:>6}")
    return lignes


profils_baseline = [etat.get_argument_profile(a) for a in etat.identified_arguments]
lignes_baseline = matrice_cooccurrence(profils_baseline, "baseline injectée")
print()
print(f"Avec n = {len(lignes_baseline)}, ces comptes sont des anecdotes, pas une mesure.")
print("Deux dimensions sont indépendantes au sens fort quand aucune ne prédit l'autre : il faut")
print("observer des arguments où l'une est en défaut et l'autre non (`A seul` / `B seul` non nuls)")
print("sur un échantillon assez grand pour que le hasard ne suffise pas à les produire.")
print("Ce carnet ne dispose pas de cet échantillon : l'indépendance est donc une HYPOTHÈSE de")
print("conception, jamais un résultat de ce carnet. La section 6 y ajoutera les arguments")
print("produits par les agents, sans changer cet ordre de grandeur.")


Co-occurrence des défauts — baseline injectée (n = 3 arguments)
                   paire   lus  deux  A seul  B seul  aucun
--------------------------------------------------------------
     sophismes / qualite     3     1       1       0      1
     sophismes / attaque     2     2       0       0      0
        sophismes / jtms     2     1       0       0      1
      sophismes / formel     2     0       2       0      0
       qualite / attaque     2     1       0       1      0
          qualite / jtms     2     1       0       0      1
        qualite / formel     2     0       1       0      1
          attaque / jtms     1     1       0       0      0
        attaque / formel     2     0       2       0      0
           jtms / formel     1     0       1       0      0

Avec n = 3, ces comptes sont des anecdotes, pas une mesure.
Deux dimensions sont indépendantes au sens fort quand aucune ne prédit l'autre : il faut
observer des arguments où l'une est en défaut et l'autre non (`

## 6. Remplir le même débat par les agents

La seconde partie refait le travail de la baseline avec les **agents de `argumentation_lib`** : les mêmes écrivains d'état (le plugin `StateManager`), mais alimentés par un modèle au lieu d'appels écrits à la main. L'état des agents est **séparé** de celui de la baseline, pour que la comparaison ne mélange jamais les deux producteurs.

**Ce qu'il faut pour exécuter cette partie** : un service OpenAI-compatible accessible, et trois variables — `VLLM_BASE_URL`, `VLLM_MODEL_ID`, `VLLM_API_KEY` — lues dans `ArgumentAnalysis/.env` ou `GenAI/.env`. Sans clé, la cellule suivante **s'arrête sur un message explicite** : le carnet ne se rabat jamais en silence sur la table injectée. C'est la différence entre une démonstration et un contournement.

Un tour qui appelle des outils coûte ici de l'ordre de la minute ; chaque phase imprime son nombre de tours et sa durée, et le plafond de tours n'est pas décoratif.

**Un dialogue Semantic Kernel ne montre pas ses appels d'outils.** Il ne rend visibles que les tours où le modèle répond *sans* appeler d'outil ; les appels et leurs résultats restent dans le fil de l'agent. Pour imprimer ce que les agents ont réellement appelé, la cellule du service installe donc un **filtre d'invocation** sur chaque noyau : il voit passer chaque appel, et son résultat.

In [12]:
# --- Le service LLM : lu dans l'environnement, verifie, et SANS repli silencieux ---
import os
from collections import Counter
from urllib.parse import urlsplit

import requests
from dotenv import load_dotenv
from openai import AsyncOpenAI
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.filters.filter_types import FilterTypes

NOTEBOOK_DIR = Path(".").resolve()          # ArgumentAnalysis/ quand le carnet tourne dans son dossier
GENAI_ENV = NOTEBOOK_DIR.parent.parent / "GenAI" / ".env"

# 1. Les .env locaux, s'ils existent (aucun n'est obligatoire : l'environnement peut suffire).
if (NOTEBOOK_DIR / ".env").is_file():
    load_dotenv(NOTEBOOK_DIR / ".env", override=False)
if GENAI_ENV.is_file():
    load_dotenv(GENAI_ENV, override=False)

# 2. Trois variables, aucune valeur de cle par defaut cachee dans le code.
VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://127.0.0.1:5002/v1")
VLLM_MODEL_ID = os.getenv("VLLM_MODEL_ID", "swift-1.5-27b")
VLLM_API_KEY = os.getenv("VLLM_API_KEY")

# 3. Sans cle, on s'arrete ici : la suite du carnet ne peut pas etre simulee.
if not VLLM_API_KEY:
    raise RuntimeError(
        "VLLM_API_KEY absente. Renseignez-la dans SymbolicAI/ArgumentAnalysis/.env ou "
        "MyIA.AI.Notebooks/GenAI/.env (VLLM_API_KEY=...), puis relancez ce carnet. "
        "La partie agents ne s'exécute pas sans service : il n'y a pas de repli sur la table injectée."
    )

# 4. Le service repond-il vraiment ? Un service muet doit se voir ici, pas au premier appel d'outil.
try:
    _reponse = requests.get(
        f"{VLLM_BASE_URL.rstrip('/')}/models",
        headers={"Authorization": f"Bearer {VLLM_API_KEY}"},
        timeout=10,
    )
    _reponse.raise_for_status()
    _servis = [m.get("id") for m in _reponse.json().get("data", [])]
except Exception as exc:  # noqa: BLE001 - le type de panne suffit, l'adresse ne doit pas fuir
    # Ni le texte de `exc` ni la chaine de cause ne sont repris : les exceptions de `requests`
    # recopient l'URL demandee, et ce message partirait dans une sortie committee avec le carnet.
    raise RuntimeError(
        f"Service LLM injoignable a l'adresse de VLLM_BASE_URL ({type(exc).__name__}). "
        "Vérifiez VLLM_BASE_URL — un service qui n'écoute pas sur 127.0.0.1 s'atteint par sa route LAN."
    ) from None

# 5. Ce que le carnet imprime du service : son TYPE, jamais son adresse. Le carnet est commite avec
# ses sorties ; une adresse de reseau local y resterait lisible. Loopback ou route LAN suffit a lire la suite.
_HOTE = urlsplit(VLLM_BASE_URL).hostname or ""
_SERVICE_LOCAL = _HOTE in {"127.0.0.1", "localhost", "::1"}

print(f"Service        : {'serveur local (loopback)' if _SERVICE_LOCAL else 'serveur du reseau local'}")
print(f"Modèle demandé : {VLLM_MODEL_ID}")
print(f"Modèles servis : {', '.join(sorted(m for m in _servis if m)) or '(liste vide)'}")
if _servis and VLLM_MODEL_ID not in _servis:
    print(f"[ATTENTION] {VLLM_MODEL_ID} n'est pas dans la liste servie : le premier appel échouera.")

# 6. Le service attendu par les agents de la lib. Tous les noyaux du carnet le partagent.
service_llm = OpenAIChatCompletion(
    service_id="global_llm_service",
    ai_model_id=VLLM_MODEL_ID,
    async_client=AsyncOpenAI(
        api_key=VLLM_API_KEY, base_url=VLLM_BASE_URL, timeout=600.0, max_retries=2
    ),
)
os.environ["ACTIVE_CHAT_MODEL_ID"] = VLLM_MODEL_ID
print("Service prêt — service_id = global_llm_service")


# 7. L'instrument de mesure des appels d'outils.
class JournalDesAppels:
    """Journal des outils appeles sur un noyau, lu entre deux tours.

    Un dialogue Semantic Kernel ne rend visibles que les tours SANS appel d'outil :
    les appels et leurs resultats restent dans le fil de l'agent. Un filtre
    d'invocation du noyau, lui, voit passer chaque appel.
    """

    def __init__(self):
        self.appels = []  # (outil qualifie plugin-fonction, en echec)

    async def filtrer(self, contexte, next):  # noqa: A002 -- SK passe la suite par ce mot-cle
        await next(contexte)
        en_echec = str(getattr(contexte.result, "value", "")).startswith("FUNC_ERROR")
        self.appels.append((contexte.function.fully_qualified_name, en_echec))

    def depuis(self, rang=0):
        """Appels enregistres a partir de `rang` : outils comptes, puis nombre d'appels en echec."""
        comptes = Counter(outil for outil, _ in self.appels[rang:])
        outils = [f"{outil} x{n}" if n > 1 else outil for outil, n in comptes.items()]
        return outils, sum(echec for _, echec in self.appels[rang:])


def noyau_neuf(journal):
    """Un noyau qui ne porte que le service et le journal des appels.

    Le plugin `StateManager` que la lib ajoute se lie au PREMIER etat qu'il recoit :
    un etat neuf demande donc un noyau neuf, sinon les outils ecriraient dans l'ancien.
    """
    noyau = Kernel()
    noyau.add_service(service_llm)
    noyau.add_filter(FilterTypes.FUNCTION_INVOCATION, journal.filtrer)
    return noyau

Service        : serveur du reseau local
Modèle demandé : swift-1.5-27b
Modèles servis : qwen3.6-35b-a3b, swift-1.5-27b
Service prêt — service_id = global_llm_service


### Dimension 1 par les agents : la phase informelle de la lib

`argumentation_lib._runner` fournit les agents de la phase informelle. Le **ProjectManager** y distribue les tâches ; l'**agent informel** a un prompt qui lui demande d'identifier les arguments et les sophismes, d'enregistrer chaque argument (`StateManager-add_identified_argument`), de consigner les sophismes détectés par l'un des outils que le plugin expose pour eux, puis de clore la tâche (`StateManager-mark_task_as_answered`). C'est exactement la dimension 1 de la baseline — produite par un modèle cette fois.

La cellule qui suit installe l'agent informel seul dans un `AgentGroupChat`, sur un état **séparé** de la baseline. Elle imprime, tour par tour, les outils réellement appelés (lus dans le journal du noyau), puis ce qui a réellement atterri dans l'état : arguments, sophismes, clôture de la tâche, erreurs d'outils. Le coût est affiché : tours et durée.

Le dialogue s'arrête dès que la phase a livré ses deux produits — au moins un argument et un sophisme **substantiels au sens des prédicats de la lib** — et clôt sa tâche. Sans cette condition d'arrêt, l'agent continue : sur ce même texte, une exécution antérieure du carnet, réglée sur le seul plafond, a vu l'agent utiliser ses trois tours, les deux derniers à redire que l'analyse était faite. Le plafond ne reste donc qu'une garde.

Trois points de câblage que la cellule respecte, parce que la lib ne les signale pas :

- le plugin `StateManager` est ajouté au kernel par le **premier** agent créé et lié à son état ; un second agent créé sur le même kernel **réutilise** ce plugin, il ne le remplace pas. Un seul état par kernel, donc — et un état neuf demande un kernel neuf ;
- l'état est injecté dans le prompt de chaque agent comme un **instantané pris à sa création**. Un agent créé avant que les arguments soient identifiés ne les verrait pas ;
- **la forme de ce que l'outil reçoit se vérifie.** Un outil écrivain enregistre ce qu'on lui passe, et le prédicat `argument_has_substance` juge une description par sa longueur (quatre mots au moins) et son recoupement avec le texte source : un conteneur qui cite le texte y passe. Or Semantic Kernel **réserve quatre noms de paramètres** — `kernel`, `service`, `execution_settings`, `arguments` : une fonction kernel qui déclare l'un d'eux reçoit l'objet du kernel qui porte ce nom, jamais la valeur de sa clé. L'outil de masse `StateManager-add_identified_arguments`, exposé à côté du singulier, déclarait un paramètre `arguments` ; appelé, il enregistrait l'instantané de l'état suivi de la liste convertie en chaîne, et répondait « OK ». La lib de la série le surcharge (`GuardedStateManagerPlugin`) avec un paramètre `descriptions`. La cellule mesure donc la **forme** de chaque description écrite : c'est elle qui dit ce qui est réellement entré dans l'état.


In [13]:
# --- Dimension 1 par les agents : la phase informelle de la lib (agent informel seul) ---
import ast
import json
import time

from pydantic import PrivateAttr
from semantic_kernel.agents import AgentGroupChat
from semantic_kernel.agents.strategies.termination.termination_strategy import (
    TerminationStrategy,
)
from semantic_kernel.contents import AuthorRole, ChatHistory, ChatMessageContent

from argumentation_lib._runner import PHASE_TASK_DESCRIPTIONS, create_informal_agent
from argumentation_lib._verdict import (
    count_substantive_arguments,
    count_substantive_fallacies,
)


class ArretPhaseInformelle(TerminationStrategy):
    """Termine le dialogue des que la phase a livre ses deux produits et clore sa tache.

    Les deux produits s'apprecient avec les predicats de la lib : au moins un
    argument et un sophisme substantiels. Le plafond de tours ne reste qu'une garde.
    """

    _etat = PrivateAttr()
    _tache = PrivateAttr()

    def __init__(self, etat, tache, **kwargs):
        super().__init__(**kwargs)
        self._etat = etat
        self._tache = tache

    async def should_agent_terminate(self, agent, history):
        return (
            self._tache in self._etat.answers
            and count_substantive_arguments(self._etat) > 0
            and count_substantive_fallacies(self._etat) > 0
        )


def _litteral(texte):
    """Lit une valeur comme litteral Python (repr de `str()`) ou comme JSON ; sinon rend le texte."""
    for lecteur in (ast.literal_eval, json.loads):
        try:
            return lecteur(texte)
        except (ValueError, SyntaxError, MemoryError, RecursionError):
            continue
    return texte


def forme_de(valeur):
    """Forme de la valeur ecrite pour un argument : 'texte', ou le conteneur recu a sa place.

    Un outil ecrivain enregistre ce qu'il recoit ; s'il recoit un objet ou une liste
    la ou le texte de l'argument est attendu, l'etat le stocke par `str(...)` -- un
    `repr` Python, a guillemets simples. Un conteneur deja serialise par le modele
    arrive en JSON. Les deux se lisent, et c'est la forme qui compte, pas le dialecte.
    """
    objet = _litteral(str(valeur).strip())
    if isinstance(objet, dict):
        return "objet JSON"
    if isinstance(objet, (list, tuple)):
        return "liste JSON"
    return "texte"


def hors_texte(etat):
    """Arguments identifies dont la description n'est pas un texte d'argument."""
    formes = {aid: forme_de(desc) for aid, desc in etat.identified_arguments.items()}
    return {aid: forme for aid, forme in formes.items() if forme != "texte"}


PLAFOND_TOURS_INFORMEL = 3  # borne le cout : chaque tour est un appel au modele

# L'etat des agents est SEPARE de la baseline `etat` : deux producteurs, deux etats.
etat_agent = UnifiedAnalysisState(DEBAT)
tache_informelle = etat_agent.add_task(PHASE_TASK_DESCRIPTIONS["informal"])
journal_agent = JournalDesAppels()
llm_kernel = noyau_neuf(journal_agent)
agent_informel = create_informal_agent(
    llm_kernel, "global_llm_service", etat_agent, task_id=tache_informelle
)

_historique = ChatHistory()
_historique.add_message(
    ChatMessageContent(role=AuthorRole.USER, content=f"Analysez le texte suivant :\n\n{DEBAT}")
)
dialogue = AgentGroupChat(
    agents=[agent_informel],
    chat_history=_historique,
    termination_strategy=ArretPhaseInformelle(
        etat_agent, tache_informelle, maximum_iterations=PLAFOND_TOURS_INFORMEL
    ),
)

tours_informel = 0
_rang = len(journal_agent.appels)
_t0 = time.perf_counter()
async for _reponse in dialogue.invoke():
    outils, echecs = journal_agent.depuis(_rang)
    _rang = len(journal_agent.appels)
    tours_informel += 1
    _echecs = f", en échec {echecs}" if echecs else ""
    print(f"[tour {tours_informel}] {_reponse.name} -> {', '.join(outils) or '(aucun outil)'}"
          f" -- arguments {len(etat_agent.identified_arguments)},"
          f" sophismes {len(etat_agent.identified_fallacies)}{_echecs}"
          f" ({time.perf_counter() - _t0:.0f} s)")

print()
print(f"Tours            : {tours_informel} / plafond {PLAFOND_TOURS_INFORMEL}")
print(f"Durée            : {time.perf_counter() - _t0:.1f} s")
print(f"Tâche clôturée   : {tache_informelle in etat_agent.answers}")
print(f"Substantifs (prédicats de la lib) : {count_substantive_arguments(etat_agent)} argument(s),"
      f" {count_substantive_fallacies(etat_agent)} sophisme(s)")
_hors_texte = hors_texte(etat_agent)
print(f"Descriptions hors texte d'argument : {len(_hors_texte)}"
      f" / {len(etat_agent.identified_arguments)}")
for _aid, _forme in _hors_texte.items():
    print(f"  {_aid} [{_forme}] : {str(etat_agent.identified_arguments[_aid])[:90]}")
if _hors_texte:
    print("[ATTENTION] Des descriptions ne sont pas des textes d'argument : elles restent"
          " telles quelles, et la comparaison ne portera que sur les arguments appariés.")

print()
print(f"Arguments identifiés ({len(etat_agent.identified_arguments)}) :")
for _aid, _desc in etat_agent.identified_arguments.items():
    print(f"  {_aid}: {str(_desc)[:110]}")
print(f"Sophismes identifiés ({len(etat_agent.identified_fallacies)}) :")
for _fid, _info in etat_agent.identified_fallacies.items():
    print(f"  {_fid}: {_info.get('type')} -> cible {_info.get('target_argument_id')}"
          f"  (famille : {_info.get('family', 'non renseignée')})")
print(f"Erreurs d'outils : {len(etat_agent.errors)}")
for _err in list(etat_agent.errors)[:5]:
    print(f"  {json.dumps(_err, ensure_ascii=False)[:200]}")


[tour 1] InformalAgent -> StateManager-add_identified_arguments, StateManager-add_identified_fallacies, StateManager-mark_task_as_answered -- arguments 3, sophismes 3 (53 s)

Tours            : 1 / plafond 3
Durée            : 52.8 s
Tâche clôturée   : True
Substantifs (prédicats de la lib) : 3 argument(s), 3 sophisme(s)
Descriptions hors texte d'argument : 0 / 3

Arguments identifiés (3) :
  arg_1: tous les experts s'accordent a dire que le projet est utile, donc il faut le financer
  arg_2: soit on le finance, soit on abandonne toute innovation
  arg_3: les couts sont eleves, mais le benefice attendu justifie l'investissement
Sophismes identifiés (3) :
  fallacy_1: appel à l'autorité -> cible arg_1  (famille : ad verecundiam)
  fallacy_2: fausse dichotomie -> cible arg_2  (famille : dilemme faux)
  fallacy_3: prémisse non étayée (appel à l'espérance) -> cible arg_3  (famille : non sequitur)
Erreurs d'outils : 0


### Dimensions 2 et 3 par les agents : un évaluateur qui pilote les écrivains de la lib

La lib n'a pas d'agent dédié à la qualité et aux contre-arguments. Elle a les **écrivains** — `StateManager-add_quality_score`, `StateManager-add_counter_argument` — et c'est tout. Un carnet qui veut ces dimensions produites par un modèle doit donc apporter l'agent : un évaluateur local, construit **après** la phase informelle (pour que ses consignes portent les arguments que celle-ci a réellement identifiés), dont les consignes reprennent le contrat des écrivains, et à qui l'appel d'outils est **imposé**. La couture est nette : *la lib fournit le contrat d'écriture, le carnet fournit celui qui juge*.

Un point de contrat se paie cher si on l'ignore : `overall` est la **somme** des notes de vertus, pas une moyenne ni une note sur 1 (contrat d'unité #1942). Or la description de l'outil annonce encore « overall (0-1) ». Les consignes de l'évaluateur énoncent donc la somme explicitement, et la cellule **revérifie** la somme sur chaque score écrit, plutôt que de faire confiance à la description de l'outil ou au modèle.

Trois détails de mécanique, tous mesurables dans la sortie :

- **la condition d'arrêt se lit dans l'état, pas dans un nombre de tours.** La stratégie de terminaison par défaut ne termine jamais d'elle-même ; celle du dialogue vérifie, elle, que chaque argument identifié porte son score de qualité — et comme l'appel d'outil y est rendu obligatoire (`FunctionChoiceBehavior.Required`), chaque tour est un coût certain. Le plafond ne reste donc qu'une garde, jamais le critère ;
- **le budget se compte en allers-retours, pas en appels.** `maximum_auto_invoke_attempts` borne les allers-retours où le modèle appelle des outils — et un aller-retour peut en porter plusieurs. Deux par argument identifié, plafonnés : un nombre écrit en dur serait faux dès que la phase informelle trouve un argument de plus ;
- **ce qui a été appelé se lit hors du dialogue.** Le journal installé sur le noyau (section 6) est relu entre deux tours : il donne les outils réellement appelés et leurs échecs, que le dialogue ne montre pas — y compris un outil que les consignes ne listent pas : une consigne ne ferme pas l'accès aux autres outils du plugin. La cellule relit donc aussi l'état après le dialogue : ce qu'un appel hors liste y a laissé se voit dans les comptes.

In [14]:
# --- Dimensions 2 et 3 par les agents : un evaluateur qui pilote les ecrivains de la lib ---
import json

from pydantic import PrivateAttr
from semantic_kernel.agents import AgentGroupChat, ChatCompletionAgent
from semantic_kernel.agents.strategies.termination.termination_strategy import (
    TerminationStrategy,
)
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior

from argumentation_lib._runner import _execution_settings_for, _state_arguments

# L'outil annonce "overall (0-1)" ; le contrat reel (#1942) est la SOMME des vertus.
# Les consignes l'ecrivent, et la verification en bas de cellule le controle.
_recensement = "\n".join(
    f"- `{aid}` : {desc}" for aid, desc in etat_agent.identified_arguments.items()
)
CONSIGNES_EVALUATEUR = f"""Vous êtes un évaluateur d'arguments. Le texte du débat est dans le premier message.
La phase informelle a identifié les arguments suivants (identifiant : texte) :
{_recensement}

Outils à utiliser (plugin StateManager), et aucun autre :
- `add_quality_score(arg_id, scores_json, overall)` : `scores_json` est un objet JSON
  {{vertu: note dans [0, 1]}}. `overall` est la SOMME des notes de vertus évaluées
  (contrat #1942) : PAS une moyenne, PAS une note sur 1.
- `add_counter_argument(original_arg, counter_content, strategy, score, target_arg_id)` :
  `original_arg` est le texte de l'argument contesté ; `strategy` parmi `contester_source`,
  `contester_inference`, `denoncer_faux_dilemme` ; `score` est la force, dans [0, 1] ;
  `target_arg_id` est l'identifiant de l'argument visé.

Contrat d'appel OBLIGATOIRE :
1. Pour CHAQUE argument listé, appelez `add_quality_score` avec les trois vertus
   `clarte`, `fondement`, `pertinence` (chacune dans [0, 1]) et `overall` égal à leur somme.
2. Pour chaque argument attaquable, appelez `add_counter_argument` — UN seul par argument.
3. Chaque appel se fait UNE SEULE FOIS : ne réécrivez pas un score déjà donné."""


class ArretQualiteComplete(TerminationStrategy):
    """Termine le dialogue des que chaque argument identifie porte son score de qualite.

    Comme l'appel d'outil est ici force a chaque reponse, un tour de plus coute un
    appel au modele et des appels d'outils ; la condition d'arret se lit donc dans
    l'etat, des que le produit attendu est entier. Le plafond ne reste qu'une garde.
    """

    _etat = PrivateAttr()

    def __init__(self, etat, **kwargs):
        super().__init__(**kwargs)
        self._etat = etat

    async def should_agent_terminate(self, agent, history):
        identifies = set(self._etat.identified_arguments)
        return bool(identifies) and identifies <= set(self._etat.argument_quality_scores)


PLAFOND_TOURS_EVALUATEUR = 3
# `maximum_auto_invoke_attempts` compte les ALLERS-RETOURS ou le modele appelle des outils
# (chacun peut en porter plusieurs), pas les appels eux-memes. Deux par argument identifie,
# plafonne : un nombre ecrit en dur serait faux des que la phase informelle trouve un
# argument de plus.
ALLERS_RETOURS_OUTILS_MAX = min(12, 2 * max(1, len(etat_agent.identified_arguments)))
n_arguments_avant = len(etat_agent.identified_arguments)

_t0 = time.perf_counter()
evaluateur = ChatCompletionAgent(
    kernel=llm_kernel,
    name="EvaluateurQualite",
    instructions=CONSIGNES_EVALUATEUR,
    service=llm_kernel.get_service("global_llm_service"),
    arguments=_state_arguments(
        etat_agent, _execution_settings_for(llm_kernel, "global_llm_service")
    ),
    function_choice_behavior=FunctionChoiceBehavior.Required(
        auto_invoke=True, maximum_auto_invoke_attempts=ALLERS_RETOURS_OUTILS_MAX
    ),
)
_historique_eval = ChatHistory()
_historique_eval.add_message(
    ChatMessageContent(role=AuthorRole.USER, content=f"Texte du débat :\n\n{DEBAT}")
)
dialogue_eval = AgentGroupChat(
    agents=[evaluateur],
    chat_history=_historique_eval,
    termination_strategy=ArretQualiteComplete(
        etat_agent, maximum_iterations=PLAFOND_TOURS_EVALUATEUR
    ),
)

journal = journal_agent  # le journal du noyau des agents, ouvert depuis la phase informelle
_sophismes_avant_eval = dict(etat_agent.identified_fallacies)
tours_evaluateur = 0
_rang = len(journal.appels)
async for _reponse in dialogue_eval.invoke():
    outils, echecs = journal.depuis(_rang)
    _rang = len(journal.appels)
    tours_evaluateur += 1
    _echecs = f", en échec {echecs}" if echecs else ""
    print(f"[tour {tours_evaluateur}] {', '.join(outils) or '(aucun outil)'}"
          f" -- scores {len(etat_agent.argument_quality_scores)}/{n_arguments_avant},"
          f" contre-arguments {len(etat_agent.counter_arguments)}{_echecs}"
          f" ({time.perf_counter() - _t0:.0f} s)")
duree_evaluation = time.perf_counter() - _t0

print()
print(f"Tours            : {tours_evaluateur} / plafond {PLAFOND_TOURS_EVALUATEUR}")
print(f"Durée            : {duree_evaluation:.1f} s")
print(f"Scores écrits    : {len(etat_agent.argument_quality_scores)}"
      f" / {n_arguments_avant} arguments")
for _aid, _sc in etat_agent.argument_quality_scores.items():
    _vertus = _sc.get("scores") or {}
    _somme = round(sum(float(_v) for _v in _vertus.values()), 4)
    _est_somme = abs(float(_sc.get("overall", 0.0)) - _somme) < 1e-6
    print(f"  {_aid}: vertus={_vertus}  overall={_sc.get('overall')}"
          f"  somme={_somme}  {'OK' if _est_somme else 'ECART : overall != somme'}")
print(f"Contre-arguments ({len(etat_agent.counter_arguments)}) :")
for _ca in etat_agent.counter_arguments:
    print(f"  {_ca.get('id')} [{_ca.get('strategy')}] force={_ca.get('score')}"
          f" -> cible {_ca.get('target_arg_id')}  {(_ca.get('counter_content') or '')[:70]}")
_sophismes_eval = {fid: f for fid, f in etat_agent.identified_fallacies.items()
                   if fid not in _sophismes_avant_eval}
_note_sophismes = ("   (outil hors de la liste de ses consignes -- le journal du tour le montre)"
                   if _sophismes_eval else "")
print(f"Sophismes ajoutés par l'évaluateur : {len(_sophismes_eval)}{_note_sophismes}")
for _fid, _info in _sophismes_eval.items():
    print(f"  {_fid}: {_info.get('type')} -> cible {_info.get('target_argument_id')}"
          f"  (famille : {_info.get('family', 'non renseignée')})")
print(f"Arguments ajoutés par l'évaluateur : "
      f"{len(etat_agent.identified_arguments) - n_arguments_avant}")
print(f"Erreurs d'outils : {len(etat_agent.errors)}")
for _err in list(etat_agent.errors)[:5]:
    print(f"  {json.dumps(_err, ensure_ascii=False)[:200]}")


[tour 1] StateManager-add_quality_score x3, StateManager-add_counter_argument x3, StateManager-set_final_conclusion, StateManager-get_current_state_snapshot x2, StateManager-mark_task_as_answered -- scores 3/3, contre-arguments 3 (79 s)

Tours            : 1 / plafond 3
Durée            : 78.6 s
Scores écrits    : 3 / 3 arguments
  arg_1: vertus={'clarte': 0.8, 'fondement': 0.3, 'pertinence': 0.6}  overall=1.7  somme=1.7  OK
  arg_2: vertus={'clarte': 0.7, 'fondement': 0.2, 'pertinence': 0.4}  overall=1.3  somme=1.3  OK
  arg_3: vertus={'clarte': 0.7, 'fondement': 0.5, 'pertinence': 0.7}  overall=1.9  somme=1.9  OK
Contre-arguments (3) :
  ca_1 [contester_source] force=0.8 -> cible arg_1  L'accord d'experts sur l'utilité d'un projet ne constitue pas, à lui s
  ca_2 [denoncer_faux_dilemme] force=0.85 -> cible arg_2  Il s'agit d'un faux dilemme : la réalité offre des options intermédiai
  ca_3 [contester_inference] force=0.6 -> cible arg_3  Le bénéfice est qualifié d'« attendu », c'est-à

### Dimensions 4 et 5 : ce que les agents ne produisent pas encore

Il reste deux dimensions dans le profil : les croyances JTMS et les résultats formels par argument. La cellule suivante les **sonde** au lieu de les supposer acquises. Elle fait trois mesures :

1. elle appelle les deux outils JTMS exposés par le plugin et regarde ce qu'ils rendent ;
2. elle compte ce que l'état des agents contient réellement sur les deux axes, et le compare à la baseline ;
3. elle croise ces comptes avec les outils que les prompts de la lib demandent : un outil peut exister dans le plugin et n'être demandé par aucun prompt.

Ce que la mesure établit n'est pas un défaut à corriger dans ce carnet : c'est la **frontière actuelle du remplissage par les agents**, écrite noir sur blanc plutôt que masquée par une table injectée. Un profil rempli par les agents n'a donc pas la même couverture qu'un profil rempli à la main — et c'est précisément ce que la colonne `couv` de la section 3 permet de dire sans mentir.


In [15]:
# --- Dimensions 4 et 5 : ce que les agents ne produisent pas encore (mesure, pas supposition) ---
from argumentation_lib._runner import PROMPT_INFORMAL_ANALYSIS, PROMPT_LOGIC_ANALYSIS
from argumentation_lib._state_manager_plugin import StateManagerPlugin

# 1. Les deux outils JTMS de la lib, appeles pour ce qu'ils rendent. Un retour
#    `FUNC_ERROR` dit un appel reste sans effet : il est compte, puis affiche sous
#    [SKIP], marqueur d'un appel ignore -- ici par le mode pedagogique de la lib.
_sonde = StateManagerPlugin(etat_agent)
_croyances_avant = len(etat_agent.jtms_beliefs)
_erreurs_avant = len(etat_agent.errors)
_retour_croyance = _sonde.jtms_create_belief(
    belief_name="essai 08e", agent_source="carnet 08e", confidence=0.8
)
_retour_justification = _sonde.jtms_add_justification(
    in_list=["a"], out_list=[], conclusion="b", agent_source="carnet 08e"
)
print("Outils JTMS de la lib :")
print(f"  jtms_create_belief      -> {_retour_croyance}")
print(f"  jtms_add_justification  -> {_retour_justification}")
_appels_sans_effet = sum(
    str(_r).startswith("FUNC_ERROR") for _r in (_retour_croyance, _retour_justification)
)
if _appels_sans_effet:
    print(f"[SKIP] {_appels_sans_effet} appel(s) JTMS sur 2 restés sans effet :"
          " refus du mode pédagogique de la lib, attendu ici")
print(f"  croyances avant/après   : {_croyances_avant} / {len(etat_agent.jtms_beliefs)}")
print(f"  erreurs d'état ajoutées : {len(etat_agent.errors) - _erreurs_avant}")

# 2. Ce que chaque etat contient sur les deux axes.
print()
print(f"{'axe':>10} {'baseline':>9} {'agents':>7}")
print("-" * 30)
print(f"{'JTMS':>10} {len(etat.jtms_beliefs):>9} {len(etat_agent.jtms_beliefs):>7}")
print(f"{'formel':>10} {len(etat.propositional_analysis_results) + len(etat.fol_analysis_results):>9}"
      f" {len(etat_agent.propositional_analysis_results) + len(etat_agent.fol_analysis_results):>7}")

# 3. Un outil expose n'est pas un outil demande : croisement plugin / prompts.
_sonde_kernel = Kernel()
_sonde_kernel.add_plugin(StateManagerPlugin(etat_agent), plugin_name="StateManager")
_outils_exposes = set(_sonde_kernel.plugins["StateManager"].functions.keys())
OUTILS_JTMS = ["jtms_create_belief", "jtms_add_justification"]
ECRIVAINS_FORMELS = ["add_propositional_analysis_result", "add_fol_analysis_result"]
OUTILS_DEBAT_LARGE = ["add_belief_set", "log_query_result"]
print()
print(f"{'outil':>36} {'exposé':>7} {'informal':>9} {'logic':>7}")
print("-" * 62)
for _outil in OUTILS_JTMS + ECRIVAINS_FORMELS + OUTILS_DEBAT_LARGE:
    print(f"{_outil:>36} {str(_outil in _outils_exposes):>7}"
          f" {str(_outil in PROMPT_INFORMAL_ANALYSIS):>9} {str(_outil in PROMPT_LOGIC_ANALYSIS):>7}")

# 4. La lecture se deduit des trois mesures ci-dessus, elle ne les precede pas.
_jtms_exposes = [o for o in OUTILS_JTMS if o in _outils_exposes]
_formels_exposes = [o for o in ECRIVAINS_FORMELS if o in _outils_exposes]
_debat_large_demandes = [
    o for o in OUTILS_DEBAT_LARGE
    if o in _outils_exposes and (o in PROMPT_INFORMAL_ANALYSIS or o in PROMPT_LOGIC_ANALYSIS)
]
print()
print("Lecture, mesure par mesure :")
print(f"- outils JTMS exposés au plugin        : {len(_jtms_exposes)} sur {len(OUTILS_JTMS)}")
print(f"- croyances écrites par ces deux appels : {len(etat_agent.jtms_beliefs) - _croyances_avant}"
      "   (ils rendent un FUNC_ERROR : le mode pédagogique de CoursIA ne charge pas de moteur JTMS)")
print(f"- écrivains formels par argument exposés : {len(_formels_exposes)} sur {len(ECRIVAINS_FORMELS)}"
      + ("   (aucun prompt ne peut les faire appeler)" if not _formels_exposes else ""))
print(f"- outils formels exposés ET demandés par un prompt de la lib : {_debat_large_demandes}")
print("  Ils écrivent des croyances et un journal de requêtes DÉBAT-LARGE, sans `arg_id` :")
print("  le profil, qui est projeté par argument, ne peut donc rien en tirer.")
print()
print("Dimensions 4 et 5 restent, aujourd'hui, du ressort de la baseline : ce n'est pas un")
print("oubli du carnet, c'est la frontière mesurée du remplissage par les agents.")


JTMS inactif (mode pédagogique) : creating JTMS belief 'essai 08e' ignoré


JTMS inactif (mode pédagogique) : adding JTMS justification ignoré


Outils JTMS de la lib :
  jtms_create_belief      -> FUNC_ERROR: Error creating belief: JTMS not available in CoursIA pedagogical mode
  jtms_add_justification  -> FUNC_ERROR: Error adding justification: JTMS not available in CoursIA pedagogical mode
[SKIP] 2 appel(s) JTMS sur 2 restés sans effet : refus du mode pédagogique de la lib, attendu ici
  croyances avant/après   : 0 / 0
  erreurs d'état ajoutées : 0

       axe  baseline  agents
------------------------------
      JTMS         2       0
    formel         3       0

                               outil  exposé  informal   logic
--------------------------------------------------------------
                  jtms_create_belief    True     False   False
              jtms_add_justification    True     False   False
   add_propositional_analysis_result   False     False   False
             add_fol_analysis_result   False     False   False
                      add_belief_set    True     False    True
                    log_qu

### Comparaison agent contre baseline

Les deux producteurs ont rempli deux états séparés. La cellule suivante les met côte à côte : ce que chacun laisse dans l'état par dimension, l'appariement des arguments d'un état à l'autre (similarité de Jaccard sur les mots, les reformulations du modèle ne recopiant pas le texte), puis le verdict de chaque argument **avec sa couverture**.

Le tableau se lit avec son seuil : chez les agents, le contre-argument le plus fort d'`arg_3` vaut 0,6, sous le seuil de 0,7 que ce carnet s'est donné — l'attaque n'est donc pas un défaut. `arg_3` est pourtant MIXTE, et c'est une autre dimension qui le porte : la phase informelle lui a attaché un sophisme. Au seuil de 0,6, l'attaque basculerait aussi, sans changer le verdict. La force vient du modèle ; le seuil est un choix du carnet, pas une prescription de la lib.

La ligne `sophismes` compte des objets écrits, pas des défauts : la baseline les pose à la main, la phase informelle les trouve dans le texte, et les deux colonnes peuvent afficher le même total sans compter la même chose. La cellule imprime à part ce que l'évaluateur a ajouté, parce qu'un outil écrivain enregistre ce qu'on lui passe sans dédoublonner : un appel hors de sa liste de consignes ferait monter ce compte sans nouveau défaut. Un compte de sophismes ne se lit donc pas comme un compte de défauts ; c'est la cible du sophisme qui décide quel argument est en défaut, pas le total.

Elle termine en réunissant les deux échantillons dans la matrice de co-occurrence de la section 5 : l'échantillon s'agrandit, le statut de l'indépendance ne change pas.

In [16]:
# --- Comparaison agent contre baseline : ce qui est produit, ce qui ne l'est pas ---
import re


def _mots(txt):
    """Mots de quatre caractères et plus, pour un appariement robuste aux reformulations."""
    return {m for m in re.findall(r"[0-9A-Za-zÀ-ÿ]{4,}", (txt or "").lower())}


def apparier(description, candidats, seuil=0.3):
    """Meilleur appariement par Jaccard ; None si le meilleur score reste sous le seuil."""
    cible = _mots(description)
    meilleur, score = None, 0.0
    for aid, autre in candidats.items():
        union = cible | _mots(autre)
        j = len(cible & _mots(autre)) / len(union) if union else 0.0
        if j > score:
            meilleur, score = aid, j
    return (meilleur if score >= seuil else None), round(score, 2)


COMPTEURS = {
    "sophismes": lambda s: len(s.identified_fallacies),
    "scores de qualité": lambda s: len(s.argument_quality_scores),
    "contre-arguments": lambda s: len(s.counter_arguments),
    "croyances JTMS": lambda s: len(s.jtms_beliefs),
    "résultats formels": lambda s: len(s.propositional_analysis_results) + len(s.fol_analysis_results),
}

print("Ce que chaque producteur laisse réellement dans l'état :")
print(f"{'dimension':>20} {'baseline':>9} {'agents':>8}")
print("-" * 40)
for nom, compte in COMPTEURS.items():
    print(f"{nom:>20} {compte(etat):>9} {compte(etat_agent):>8}")
print(f"{'arguments':>20} {len(etat.identified_arguments):>9} {len(etat_agent.identified_arguments):>8}")
print()

print("Appariement des arguments de la baseline avec ceux des agents (Jaccard sur les mots) :")
paires = {}
for aid, desc in etat.identified_arguments.items():
    jumeau, j = apparier(desc, etat_agent.identified_arguments)
    paires[aid] = (jumeau, j)
    print(f"  {aid:>15} -> {str(jumeau):>15}  (Jaccard {j})")
print()

print("Verdict de chaque argument, par les deux producteurs :")
print(f"{'baseline':>15} {'verdict':>9} {'couv':>5}   {'agent':>15} {'verdict':>9} {'couv':>5}")
print("-" * 70)
couvertures_agents = []
for aid, (jumeau, _j) in paires.items():
    vb, _db, cb, _ = verdict_multidim(etat.get_argument_profile(aid))
    if jumeau:
        va, _da, ca, _ = verdict_multidim(etat_agent.get_argument_profile(jumeau))
        couvertures_agents.append(ca)
        print(f"{aid:>15} {vb:>9} {cb:>5}   {jumeau:>15} {va:>9} {ca:>5}")
    else:
        print(f"{aid:>15} {vb:>9} {cb:>5}   {'-':>15} {'-':>9} {'-':>5}")
print()
# La phrase de lecture se calcule sur les verdicts ci-dessus : elle ne precise rien a leur place.
print(f"`couv` reste le juge : les agents lisent au plus {max(couvertures_agents) if couvertures_agents else 0}"
      f" dimension(s) d'un argument apparié, la baseline"
      f" {max(verdict_multidim(etat.get_argument_profile(a))[2] for a in etat.identified_arguments)}.")
print("Un verdict ne se compare d'un producteur à l'autre qu'avec sa couverture.")

print()
lignes_union = matrice_cooccurrence(
    profils_baseline
    + [etat_agent.get_argument_profile(a) for a in etat_agent.identified_arguments],
    "baseline + agents",
)
print()
print(f"L'échantillon réuni fait n = {len(lignes_union)}, contre {len(profils_baseline)} pour la")
print("baseline seule : un échantillon plus grand, pas un changement de statut — l'indépendance")
print("des dimensions reste une hypothèse de conception.")
_agent_a_formel = (
    len(etat_agent.jtms_beliefs)
    + len(etat_agent.propositional_analysis_results)
    + len(etat_agent.fol_analysis_results)
) > 0
if not _agent_a_formel:
    print("Les paires avec `jtms` ou `formel` sont celles de la section 5 : ces deux dimensions")
    print("ne sont lues que pour la baseline, et `lus` ne compte un argument que lorsque les")
    print("deux dimensions de la paire y sont lues.")


Ce que chaque producteur laisse réellement dans l'état :
           dimension  baseline   agents
----------------------------------------
           sophismes         3        3
   scores de qualité         3        3
    contre-arguments         3        3
      croyances JTMS         2        0
   résultats formels         3        0
           arguments         3        3

Appariement des arguments de la baseline avec ceux des agents (Jaccard sur les mots) :
            arg_1 ->           arg_1  (Jaccard 0.38)
            arg_2 ->           arg_2  (Jaccard 0.71)
            arg_3 ->           arg_3  (Jaccard 0.7)

Verdict de chaque argument, par les deux producteurs :
       baseline   verdict  couv             agent   verdict  couv
----------------------------------------------------------------------
          arg_1    FAIBLE     5             arg_1    FAIBLE     3
          arg_2    FAIBLE     4             arg_2    FAIBLE     3
          arg_3    SOLIDE     3             arg_3  

## 7. Exercices

### Exercice 1 — Ajouter un quatrième argument et son profil
Ajoutez un argument `a_risque` (« le projet est risqué donc à éviter ») au débat. Scorez-le (faible fondement car aucun risque chiffré), attachez-lui le sophisme `appel_a_la_peur`. Construire son profil et vérifier qu'il apparaît dans `get_fallacious_arguments()`.
- **Indice :** `etat.add_argument(...)`, `etat.add_fallacy(..., target_arg_id=a_risque)`, `etat.add_quality_score(a_risque, ...)`, puis `etat.get_argument_profile(a_risque)`.
- **Question de lecture :** quelle est la couverture (`couv`) de ce nouvel argument, et que faudrait-il lui ajouter pour la porter à cinq ?


In [17]:
# Exercice 1 a completer
# TODO etudiant : ajouter a_risque, le scorer, lui attacher un sophisme, construire son profil.
print("Exercice 1 à compléter : profil d'un quatrième argument fallacieux.")


Exercice 1 à compléter : profil d'un quatrième argument fallacieux.


### Exercice 2 — Seuil de faiblesse : analyse de sensibilité
Faites varier le `threshold` de `get_weak_arguments` entre 0.3 et 0.8 par pas de 0.1 (le threshold est une **fraction** dans [0, 1] — contrat #1942 : une valeur > 1.0 lève `ValueError`). Pour chaque seuil, affichez combien d'arguments sont jugés faibles. À partir de quel seuil `a_fauxdilemme` (fraction 1.55/3 ≈ 0.52) bascule-t-il dans la catégorie faible ?
- **Indice :** bouclez sur les seuils, appelez `get_weak_arguments(threshold=s)`, affichez `len(...)`. Le basculement se produit quand `threshold > 1.55/3 ≈ 0.52`.


In [18]:
# Exercice 2 a completer
# TODO etudiant : balayer threshold, tracer le compte d'arguments faibles vs seuil.
print("Exercice 2 à compléter : sensibilité du seuil de faiblesse.")


Exercice 2 à compléter : sensibilité du seuil de faiblesse.


### Exercice 3 — Verdict agrégé pondéré, et ce qu'il perd

Implémentez une fonction `verdict_global(profil)` qui combine les dimensions en un **score unique** : par exemple `fraction_qualite * 0.4 + (1 - n_sophismes/3) * 0.3 + (1 - max_force_contre) * 0.3` où `fraction_qualite = overall / len(scores)`. Calculez ce score pour les trois arguments et vérifiez qu'il classe `a_cout` au-dessus d'`a_expert`.
- **Indice :** `profil.quality_score['overall'] / len(profil.quality_score['scores'])`, `len(profil.fallacies)`, `max(c['score'] for c in profil.counter_arguments)`. Les poids sont une **décision de design** — justifiez-les.
- **Question de lecture :** comparez le classement obtenu à celui de `verdict_multidim`. Que gagne un score unique (un ordre total, comparable entre débats) et que perd-il (la couverture : un argument noté sur deux dimensions et un argument noté sur cinq deviennent indistinguables) ? Traitez les dimensions manquantes explicitement (`None`) au lieu de les compter comme zéro.


In [19]:
# Exercice 3 a completer
# TODO etudiant : verdict_global(profil) combine les dimensions, classer les 3 arguments.
print("Exercice 3 à compléter : verdict agrégé pondéré des 5 dimensions.")


Exercice 3 à compléter : verdict agrégé pondéré des 5 dimensions.


## 8. Ponts avec la série

| Direction | Lien | Relation |
|-----------|------|----------|
| <-> Agentic-1-informal | [Détection de sophismes par taxonomie](Argumentation-02-Fallacies-Detection-Python.ipynb) | Couvre la **dimension 1** (sophismes). Ce notebook la réunit avec les 4 autres dans le profil. |
| <-> Dung AF Semantics | [Sémantiques de Dung](Argumentation-03-Dung-AF-Semantics-Python.ipynb) | Voisin conceptuel — les frameworks Dung vivent dans `dung_frameworks` (comptes dans les statistiques globales) et ne sont **pas** projetés dans le profil par argument (seuls les verdicts PL/FOL de la dimension 5 y entrent). |
| <-> Restitution 3 Actes | [Restitution narrative](Argumentation-08d-Restitution-3-Actes-Python.ipynb) | Le profil est l'unité de base restituée dans le récit en 3 actes (un argument = un personnage). |

## 9. Conclusion

`ArgumentProfile` est la **vue projetée** qui transforme un état d'analyse multidimensionnel en une fiche exploitable **par argument**. Sans lui, chaque dimension vivrait en silo ; avec lui, un agent peut raisonner sur un argument précis en voyant d'un coup d'œil ses forces et faiblesses sur les axes **réellement renseignés**.

**Leçon centrale** : le profil rend un verdict **avec sa couverture**. Deux dimensions peuvent diverger — un argument formellement satisfiable peut être rhétoriquement fallacieux — mais l'indépendance des cinq dimensions est une **hypothèse de conception** que ce carnet ne mesure pas : trois arguments injectés, plus ceux que les agents identifient, ne suffisent pas à l'établir. Ce que le carnet mesure, en revanche, c'est la couverture : quelles dimensions sont renseignées, par qui, et ce qu'un verdict sur trois axes ne dit pas des deux autres.

**Note d'honnêteté (G.2)** : la première partie du carnet est une **baseline** injectée déterministement, sans appel LLM ; la seconde partie est réellement exécutée par les agents de `argumentation_lib` contre un service OpenAI-compatible. Leur comparaison situe le remplissage par les agents aujourd'hui : dimension 1 par l'agent informel de la lib ; dimensions 2 et 3 par un évaluateur local qui pilote les écrivains de la lib ; dimensions 4 et 5 non produites — les outils JTMS rendent une erreur en mode pédagogique, et les écrivains formels par argument ne sont pas exposés comme outils. Le contrat du profil est invariant : il accepte les cinq dimensions de la même façon, quel que soit celui qui les remplit.

Le carnet suivant est la [Synthèse multidimensionnelle](Argumentation-08f-Synthese-Multidimensionnelle-Python.ipynb) : le catalogue light/standard/full qui orchestre l'ensemble de ces dimensions et en restitue la conclusion en trois actes.

## Références et provenance

**Cadre de production** : [EPIC #4960 — Volet B, étape 3](https://github.com/jsboige/CoursIA/issues/4960), landing de l’essence EPITA-IS.
